# Unit19_Example_Catalyst_Diffusion_Reaction | PINN 應用實例：多孔觸媒顆粒內擴散-反應

本 Notebook 以 **TensorFlow/Keras 從零實作一個物理資訊神經網路(PINN)**，求解化學反應工程的經典邊界值問題(BVP)：**多孔觸媒顆粒內的擴散-反應**，並計算**觸媒有效因子(effectiveness factor)**。此問題源自 Thiele (1939) 的開創性工作，具有**解析解**可供驗證。

## 目標
- 以 PINN 求解二階常微分方程 BVP： $\dfrac{d^2 C}{d\xi^2} = \phi^2 C$ （無因次形式）
- 用**巢狀自動微分**計算**二階導數**，並同時滿足 **Dirichlet(表面)** 與 **Neumann(中心對稱)** 邊界條件
- **正問題**：已知 Thiele 模數 $\phi$ ，求解顆粒內濃度分佈，並計算有效因子 $\eta = \tanh\phi / \phi$
- **反問題**：由稀疏且帶雜訊的顆粒內濃度量測，**辨識 Thiele 模數** $\phi$ （進而估計有效因子）

資料背景：本範例的「真實解」來自經典反應工程理論的解析解（Thiele modulus 理論），屬可驗證的物理模型。

---
### 0. 環境設定
- 本範例以解析物理模型產生資料，無需下載外部資料集。
- 使用 TensorFlow/Keras 從零實作 PINN，資料量小，可在 Colab 或本地 CPU 環境執行。

In [1]:
from pathlib import Path
import tensorflow as tf
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'P4_Unit19_Example_Catalyst_Diffusion_Reaction'
SOURCE_DATA_DIR = 'catalyst'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

if IN_COLAB:
    shortcut_path = '/content/ChemE-3590'
    try:
        os.remove(shortcut_path)
    except FileNotFoundError:
        pass
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3590')
    os.symlink(source_path, shortcut_path)
    NOTEBOOK_DIR = Path(shortcut_path) / 'Part_4' / 'Unit19'
else:
    NOTEBOOK_DIR = Path.cwd()

OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
MODEL_DIR = OUTPUT_DIR / 'models'
FIG_DIR = OUTPUT_DIR / 'figs'
for d in (OUTPUT_DIR, MODEL_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 模型輸出目錄: {MODEL_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

print(f"\nTensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✓ 偵測到 GPU：{gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("△ 未偵測到 GPU，將使用 CPU（本範例資料量小，CPU 即可）")

✓ 偵測到 Local 環境

✓ Notebook工作目錄: D:\MyGit\ChemE-3590\Part_4\Unit19
✓ 結果輸出目錄: D:\MyGit\ChemE-3590\Part_4\Unit19\outputs\P4_Unit19_Example_Catalyst_Diffusion_Reaction
✓ 模型輸出目錄: D:\MyGit\ChemE-3590\Part_4\Unit19\outputs\P4_Unit19_Example_Catalyst_Diffusion_Reaction\models
✓ 圖檔輸出目錄: D:\MyGit\ChemE-3590\Part_4\Unit19\outputs\P4_Unit19_Example_Catalyst_Diffusion_Reaction\figs

TensorFlow Version: 2.10.1
✓ 偵測到 GPU：/physical_device:GPU:0


---
### 1. 導入必要套件

In [2]:
# 基礎套件
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow版本: {tf.__version__}")
print(f"Keras版本: {keras.__version__}")

# 設定隨機種子以確保結果可重現
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.rcParams['axes.unicode_minus'] = False

TensorFlow版本: 2.10.1
Keras版本: 2.10.0


---
### 2. 問題定義：多孔觸媒顆粒內的擴散-反應

在非均相催化中，反應物必須先**擴散**進入多孔觸媒顆粒內部，才能在活性位發生**反應**。擴散與反應的競爭決定了顆粒內部的濃度分佈與整體反應速率。

考慮一個**平板(slab)幾何**的觸媒，內部進行一級不可逆反應。經無因次化後，穩態質量平衡為：

$$
\frac{d^2 C}{d\xi^2} = \phi^2 C
$$

其中：
- $C = c / c_s$ 是無因次濃度（ $c_s$ 為顆粒外表面濃度）
- $\xi = x / L$ 是無因次位置（ $\xi=0$ 為顆粒中心， $\xi=1$ 為外表面， $L$ 為半厚度）
- $\phi = L\sqrt{k / D_e}$ 是 **Thiele 模數**，代表「反應速率 / 擴散速率」的比值

**邊界條件**：

$$
\left.\frac{dC}{d\xi}\right|_{\xi=0} = 0 \quad (\text{中心對稱，無通量}), \qquad C(1) = 1 \quad (\text{表面濃度})
$$

**解析解**（可用於驗證 PINN）：

$$
C(\xi) = \frac{\cosh(\phi\,\xi)}{\cosh(\phi)}
$$

**有效因子(Effectiveness Factor)** $\eta$ 衡量實際反應速率相對於無擴散限制時的比值，對平板一級反應：

$$
\eta = \frac{\tanh\phi}{\phi}
$$

> Thiele 模數越大代表內擴散限制越嚴重， $\eta$ 越小。這是反應器設計與觸媒選型的核心指標。

In [3]:
# Thiele 模數真值
PHI_TRUE = 2.0

def analytical_C(xi, phi=PHI_TRUE):
    """解析解 C(xi) = cosh(phi*xi)/cosh(phi)"""
    return np.cosh(phi * xi) / np.cosh(phi)

def eta_exact(phi):
    """平板一級反應有效因子 eta = tanh(phi)/phi"""
    return np.tanh(phi) / phi

xi_plot = np.linspace(0.0, 1.0, 200).reshape(-1, 1)

# 不同 Thiele 模數的濃度分佈
plt.figure(figsize=(7, 4.5))
for phi in [0.5, 1.0, 2.0, 5.0]:
    plt.plot(xi_plot, analytical_C(xi_plot, phi),
             label=f'$\\phi$={phi}, $\\eta$={eta_exact(phi):.3f}')
plt.xlabel('Dimensionless position  $\\xi$ (0=center, 1=surface)')
plt.ylabel('Dimensionless concentration  C')
plt.title('Catalyst Pellet Concentration Profiles (Analytical)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'analytical_profiles.png', dpi=120)
plt.show()

print(f"真值 Thiele 模數 phi = {PHI_TRUE}")
print(f"解析有效因子 eta = tanh(phi)/phi = {eta_exact(PHI_TRUE):.4f}")
print(f"中心濃度 C(0) = 1/cosh(phi) = {analytical_C(0.0):.4f}")

真值 Thiele 模數 phi = 2.0
解析有效因子 eta = tanh(phi)/phi = 0.4820
中心濃度 C(0) = 1/cosh(phi) = 0.2658


---
### 3. 建立 PINN 模型

PINN 的「解」是一個以**無因次位置 $\xi$ 為輸入、無因次濃度 $C$ 為輸出**的神經網路 $\hat{C}_\theta(\xi)$ 。
因為要計算**二階導數** $\dfrac{d^2 C}{d\xi^2}$ ，激活函數必須夠平滑，這裡採用 $\tanh$ 。

In [4]:
def build_pinn(n_hidden=4, n_units=20, seed=SEED):
    """以 xi 為輸入、C 為輸出的 PINN (MLP + tanh)"""
    init = keras.initializers.GlorotNormal(seed=seed)
    inp = keras.Input(shape=(1,), name='xi')
    x = inp
    for i in range(n_hidden):
        x = layers.Dense(n_units, activation='tanh',
                         kernel_initializer=init, name=f'hidden_{i+1}')(x)
    out = layers.Dense(1, kernel_initializer=init, name='C')(x)
    return keras.Model(inp, out, name='PINN_Catalyst')

pinn_demo = build_pinn()
pinn_demo.summary()

Model: "PINN_Catalyst"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 xi (InputLayer)             [(None, 1)]               0         


 hidden_1 (Dense)            (None, 20)                40        


 hidden_2 (Dense)            (None, 20)                420       


... (3 outputs left out)

Total params: 1,321


Trainable params: 1,321


Non-trainable params: 0


_________________________________________________________________


---
### 4. 正問題：二階導數與 BVP 殘差

**正問題**：已知 Thiele 模數 $\phi$ ，求解顆粒內濃度分佈。

關鍵是用**兩層巢狀 `tf.GradientTape`** 計算二階導數，組成 PDE 殘差：

$$
r(\xi) = \frac{d^2 \hat{C}_\theta}{d\xi^2} - \phi^2\,\hat{C}_\theta(\xi)
$$

同時在邊界點強制：中心 $\dfrac{dC}{d\xi}\big|_{\xi=0}=0$ （Neumann）與表面 $C(1)=1$ （Dirichlet）。

In [5]:
def pde_residual(model, xi, phi):
    """二階殘差 r = d2C/dxi2 - phi^2 * C，以巢狀自動微分計算二階導數"""
    with tf.GradientTape() as t2:
        t2.watch(xi)
        with tf.GradientTape() as t1:
            t1.watch(xi)
            C = model(xi)
        dC = t1.gradient(C, xi)
    d2C = t2.gradient(dC, xi)
    return d2C - (phi ** 2) * C

def neumann_center(model, xi0):
    """中心導數 dC/dxi |_{xi=0}，用於 Neumann BC"""
    with tf.GradientTape() as tb:
        tb.watch(xi0)
        C = model(xi0)
    return tb.gradient(C, xi0)

# 配點(collocation points) 與邊界點
N_COLLOC = 100
xi_colloc = tf.convert_to_tensor(
    np.linspace(0.0, 1.0, N_COLLOC).reshape(-1, 1), dtype=tf.float32)
xi_center = tf.constant([[0.0]], dtype=tf.float32)   # 中心 (Neumann)
xi_surf = tf.constant([[1.0]], dtype=tf.float32)     # 表面 (Dirichlet)

print(f"配點數量: {N_COLLOC}，範圍 xi ∈ [0, 1]")

配點數量: 100，範圍 xi ∈ [0, 1]


### 4.1 訓練正問題 PINN

總損失 = PDE 殘差 + 表面 Dirichlet BC + 中心 Neumann BC（邊界項加權 10 倍以強化約束）：

$$
\mathcal{L} = \mathcal{L}_{\mathrm{pde}} + 10\big(\mathcal{L}_{\mathrm{surf}} + \mathcal{L}_{\mathrm{center}}\big)
$$

In [6]:
pinn_fwd = build_pinn()
opt_fwd = keras.optimizers.Adam(learning_rate=1e-3)
W_BC = 10.0

@tf.function
def train_step_forward():
    with tf.GradientTape() as tape:
        r = pde_residual(pinn_fwd, xi_colloc, PHI_TRUE)
        loss_pde = tf.reduce_mean(tf.square(r))
        loss_surf = tf.reduce_mean(tf.square(pinn_fwd(xi_surf) - 1.0))
        loss_center = tf.reduce_mean(tf.square(neumann_center(pinn_fwd, xi_center)))
        loss = loss_pde + W_BC * (loss_surf + loss_center)
    grads = tape.gradient(loss, pinn_fwd.trainable_variables)
    opt_fwd.apply_gradients(zip(grads, pinn_fwd.trainable_variables))
    return loss, loss_pde, loss_surf, loss_center

EPOCHS_FWD = 8000
for ep in range(EPOCHS_FWD + 1):
    loss, lp, ls, lc = train_step_forward()
    if ep % 1000 == 0:
        print(f"Epoch {ep:5d} | total={float(loss):.3e} | pde={float(lp):.3e} | "
              f"surf={float(ls):.3e} | center={float(lc):.3e}")

Epoch     0 | total=7.865e+00 | pde=1.963e+00 | surf=2.802e-01 | center=3.100e-01


Epoch  1000 | total=3.664e-03 | pde=3.661e-03 | surf=2.148e-07 | center=1.531e-08


Epoch  2000 | total=1.030e-03 | pde=9.218e-04 | surf=1.038e-05 | center=4.238e-07


Epoch  3000 | total=5.956e-04 | pde=5.861e-04 | surf=4.933e-07 | center=4.542e-07


Epoch  4000 | total=3.877e-04 | pde=3.660e-04 | surf=2.161e-06 | center=1.361e-08


Epoch  5000 | total=4.232e-04 | pde=2.122e-04 | surf=1.957e-05 | center=1.526e-06


Epoch  6000 | total=3.910e-04 | pde=1.066e-04 | surf=2.527e-05 | center=3.170e-06


Epoch  7000 | total=1.338e-04 | pde=1.742e-05 | surf=3.905e-06 | center=7.734e-06


Epoch  8000 | total=1.674e-05 | pde=1.550e-05 | surf=3.193e-09 | center=1.208e-07


### 4.2 正問題結果、解析解比較與有效因子

PINN 在**不使用任何濃度資料**下，僅憑 PDE 與邊界條件重建了顆粒內濃度分佈。
我們與解析解比對，並以數值積分 $\eta = \int_0^1 C(\xi)\, d\xi$ 估計有效因子，與理論值 $\tanh\phi/\phi$ 比較。

In [7]:
xi_t = tf.convert_to_tensor(xi_plot, dtype=tf.float32)
C_pinn = pinn_fwd(xi_t).numpy()
C_ana = analytical_C(xi_plot)

rel_l2 = np.linalg.norm(C_pinn - C_ana) / np.linalg.norm(C_ana)

# 有效因子：eta = ∫_0^1 C dxi (無因次形式，速率 ∝ C，表面 C=1)
eta_pinn = np.trapz(C_pinn.ravel(), xi_plot.ravel())
eta_th = eta_exact(PHI_TRUE)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(xi_plot, C_ana, 'k-', lw=2, label='Analytical')
axes[0].plot(xi_plot, C_pinn, 'r--', lw=2, label='PINN (forward)')
axes[0].set_xlabel('Dimensionless position  $\\xi$')
axes[0].set_ylabel('Dimensionless concentration  C')
axes[0].set_title(f'Forward PINN vs Analytical  (rel. $L_2$ = {rel_l2:.2e})')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(xi_plot, np.abs(C_pinn - C_ana), 'b-')
axes[1].set_xlabel('Dimensionless position  $\\xi$')
axes[1].set_ylabel('Absolute Error')
axes[1].set_title('Pointwise Absolute Error')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'forward_pinn_result.png', dpi=120)
plt.show()

print(f"相對 L2 誤差: {rel_l2:.3e}")
print(f"有效因子 eta (PINN 積分) = {eta_pinn:.4f}")
print(f"有效因子 eta (理論 tanh/phi) = {eta_th:.4f}")
print(f"eta 相對誤差: {abs(eta_pinn - eta_th)/eta_th*100:.2f}%")

相對 L2 誤差: 4.699e-04
有效因子 eta (PINN 積分) = 0.4820
有效因子 eta (理論 tanh/phi) = 0.4820
eta 相對誤差: 0.00%


---
### 5. 反問題：Thiele 模數辨識

實務上我們常可量測顆粒內**部分位置的濃度**（例如以顯微探針或切片分析），但**擴散-反應參數未知**。
Thiele 模數 $\phi = L\sqrt{k/D_e}$ 封裝了速率常數 $k$ 與有效擴散係數 $D_e$ 。

**反問題**：由少量**帶雜訊**的顆粒內濃度量測，把 $\phi$ 設為**可訓練變數**，同時求解濃度分佈並辨識 $\phi$ 。

總損失 = 資料損失 + PDE 殘差（含未知 $\phi$ ）+ 邊界條件：

$$
\mathcal{L} = \mathcal{L}_{\mathrm{data}} + \mathcal{L}_{\mathrm{pde}}(\phi) + 10\big(\mathcal{L}_{\mathrm{surf}} + \mathcal{L}_{\mathrm{center}}\big)
$$

In [8]:
# 產生少量帶雜訊的顆粒內濃度量測
rng = np.random.default_rng(SEED)
N_DATA = 10
xi_obs = np.linspace(0.0, 1.0, N_DATA).reshape(-1, 1)
noise_std = 0.02
C_obs = analytical_C(xi_obs) + rng.normal(0, noise_std, size=xi_obs.shape)

xi_obs_tf = tf.convert_to_tensor(xi_obs, dtype=tf.float32)
C_obs_tf = tf.convert_to_tensor(C_obs, dtype=tf.float32)

plt.figure(figsize=(7, 4.5))
plt.plot(xi_plot, analytical_C(xi_plot), 'k-', lw=2, label='True profile ($\\phi$=2.0)')
plt.plot(xi_obs, C_obs, 'ro', ms=7, label=f'Noisy data (N={N_DATA})')
plt.xlabel('Dimensionless position  $\\xi$')
plt.ylabel('Dimensionless concentration  C')
plt.title('Sparse Noisy In-Pellet Measurements for Inverse Problem')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'inverse_data.png', dpi=120)
plt.show()

print(f"觀測資料點數: {N_DATA}，雜訊標準差: {noise_std}")

觀測資料點數: 10，雜訊標準差: 0.02


### 5.1 訓練反問題 PINN

初始猜測 $\phi_{\mathrm{init}}=1.0$ （遠離真值 2.0）。訓練過程中追蹤 $\phi$ 的收斂。

In [9]:
pinn_inv = build_pinn()
phi_var = tf.Variable(1.0, dtype=tf.float32, name='phi')   # 待辨識的 Thiele 模數
opt_inv = keras.optimizers.Adam(learning_rate=5e-3)

@tf.function
def train_step_inverse():
    with tf.GradientTape() as tape:
        C_pred = pinn_inv(xi_obs_tf)
        loss_data = tf.reduce_mean(tf.square(C_pred - C_obs_tf))
        r = pde_residual(pinn_inv, xi_colloc, phi_var)
        loss_pde = tf.reduce_mean(tf.square(r))
        loss_surf = tf.reduce_mean(tf.square(pinn_inv(xi_surf) - 1.0))
        loss_center = tf.reduce_mean(tf.square(neumann_center(pinn_inv, xi_center)))
        loss = loss_data + loss_pde + W_BC * (loss_surf + loss_center)
    var_list = pinn_inv.trainable_variables + [phi_var]
    grads = tape.gradient(loss, var_list)
    opt_inv.apply_gradients(zip(grads, var_list))
    return loss, loss_data, loss_pde, phi_var

EPOCHS_INV = 8000
phi_history = []
for ep in range(EPOCHS_INV + 1):
    loss, ld, lp, pv = train_step_inverse()
    if ep % 200 == 0:
        phi_history.append((ep, float(pv)))
    if ep % 1000 == 0:
        print(f"Epoch {ep:5d} | total={float(loss):.3e} | data={float(ld):.3e} | "
              f"pde={float(lp):.3e} | phi={float(pv):.4f}")

phi_pred = float(phi_var)
print(f"\n✓ 辨識結果: phi_pred = {phi_pred:.4f}  (真值 phi_true = {PHI_TRUE})")
print(f"  相對誤差: {abs(phi_pred - PHI_TRUE)/PHI_TRUE*100:.2f}%")
print(f"  由辨識 phi 推得有效因子 eta = {eta_exact(phi_pred):.4f} (真值 {eta_exact(PHI_TRUE):.4f})")

Epoch     0 | total=6.254e+00 | data=7.109e-02 | pde=2.817e-01 | phi=0.9950


Epoch  1000 | total=2.682e-02 | data=1.450e-02 | pde=2.746e-03 | phi=1.4645


Epoch  2000 | total=1.381e-03 | data=9.594e-04 | pde=2.878e-04 | phi=1.8936


Epoch  3000 | total=6.356e-04 | data=3.392e-04 | pde=1.002e-04 | phi=1.9910


Epoch  4000 | total=4.776e-04 | data=3.630e-04 | pde=4.269e-05 | phi=2.0031


Epoch  5000 | total=5.853e-04 | data=3.629e-04 | pde=1.728e-04 | phi=2.0135


Epoch  6000 | total=1.472e-03 | data=3.090e-04 | pde=1.580e-04 | phi=2.0107


Epoch  7000 | total=6.055e-03 | data=3.833e-04 | pde=2.854e-03 | phi=2.0137


Epoch  8000 | total=9.427e-03 | data=3.658e-04 | pde=4.737e-03 | phi=2.0027

✓ 辨識結果: phi_pred = 2.0027  (真值 phi_true = 2.0)
  相對誤差: 0.14%
  由辨識 phi 推得有效因子 eta = 0.4815 (真值 0.4820)


### 5.2 反問題結果： $\phi$ 收斂與濃度分佈擬合

左圖顯示可訓練 Thiele 模數 $\phi$ 從初始猜測收斂到真值；右圖顯示 PINN 解同時擬合稀疏資料並符合物理。

In [10]:
phi_eps = np.array([h[0] for h in phi_history])
phi_vals = np.array([h[1] for h in phi_history])
C_inv = pinn_inv(xi_t).numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(phi_eps, phi_vals, 'b-o', ms=3, label='Identified $\\phi$')
axes[0].axhline(PHI_TRUE, color='k', ls='--', label=f'True $\\phi$ = {PHI_TRUE}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('$\\phi$')
axes[0].set_title('Convergence of Identified Thiele Modulus')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(xi_plot, C_ana, 'k-', lw=2, label='True profile')
axes[1].plot(xi_plot, C_inv, 'g--', lw=2, label='PINN (inverse)')
axes[1].plot(xi_obs, C_obs, 'ro', ms=6, label='Noisy data')
axes[1].set_xlabel('Dimensionless position  $\\xi$')
axes[1].set_ylabel('Dimensionless concentration  C')
axes[1].set_title('Inverse PINN Fit')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'inverse_pinn_result.png', dpi=120)
plt.show()

print(f"最終 phi_pred = {phi_pred:.4f} | phi_true = {PHI_TRUE}")

最終 phi_pred = 2.0027 | phi_true = 2.0


---
### 6. 模型與結果儲存

In [11]:
# 儲存正問題與反問題 PINN 權重 (TensorFlow checkpoint 格式)
fwd_path = MODEL_DIR / 'pinn_catalyst_forward'
inv_path = MODEL_DIR / 'pinn_catalyst_inverse'
pinn_fwd.save_weights(str(fwd_path))
pinn_inv.save_weights(str(inv_path))

import json
results = {
    'phi_true': PHI_TRUE,
    'phi_pred': phi_pred,
    'eta_exact': float(eta_exact(PHI_TRUE)),
    'eta_pinn_integral': float(eta_pinn),
    'eta_from_identified_phi': float(eta_exact(phi_pred)),
    'forward_rel_L2': float(rel_l2),
}
with open(MODEL_DIR / 'catalyst_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✓ 正問題權重儲存: {fwd_path.name}")
print(f"✓ 反問題權重儲存: {inv_path.name}")
print(f"✓ 結果儲存: catalyst_results.json")
print(json.dumps(results, ensure_ascii=False, indent=2))

✓ 正問題權重儲存: pinn_catalyst_forward
✓ 反問題權重儲存: pinn_catalyst_inverse
✓ 結果儲存: catalyst_results.json
{
  "phi_true": 2.0,
  "phi_pred": 2.0027143955230713,
  "eta_exact": 0.48201379003790845,
  "eta_pinn_integral": 0.4820157827144891,
  "eta_from_identified_phi": 0.48145599568733655,
  "forward_rel_L2": 0.00046992658983666726
}


---
### 7. 總結與文獻參考

本 Notebook 以 PINN 求解化學反應工程的經典問題，完成了：

1. **問題建模**：多孔觸媒顆粒內擴散-反應 BVP $\dfrac{d^2 C}{d\xi^2}=\phi^2 C$ ，具解析解 $C(\xi)=\cosh(\phi\xi)/\cosh(\phi)$
2. **二階導數自動微分**：以兩層巢狀 `tf.GradientTape` 計算 $d^2C/d\xi^2$
3. **混合邊界條件**：同時滿足表面 Dirichlet 與中心 Neumann 條件
4. **正問題**：在**無濃度資料**下重建顆粒內分佈，並計算**有效因子** $\eta=\tanh\phi/\phi$
5. **反問題**：由稀疏帶雜訊量測**辨識 Thiele 模數** $\phi$ ，進而估計有效因子

**工程意義**：Thiele 模數與有效因子是觸媒顆粒設計、反應器放大與選型的關鍵指標。PINN 提供了一個能**同時求解分佈並由實測資料反推物性參數**的統一框架，適合製程資料稀疏但物理已知的情境。

**文獻參考**：
- Thiele, E. W. (1939). "Relation between Catalytic Activity and Size of Particle." *Industrial & Engineering Chemistry*, 31(7), 916-920.
- Fogler, H. S. *Elements of Chemical Reaction Engineering* (內擴散與有效因子章節).
- Froment, G. F., Bischoff, K. B., De Wilde, J. *Chemical Reactor Analysis and Design*, 3rd ed., Wiley.
- Raissi, M., Perdikaris, P., Karniadakis, G. E. (2019). "Physics-Informed Neural Networks." *Journal of Computational Physics*, 378, 686-707.
- Karniadakis, G. E., et al. (2021). "Physics-Informed Machine Learning." *Nature Reviews Physics*, 3, 422-440.
- Cuomo, S., et al. (2022). "Scientific Machine Learning Through Physics-Informed Neural Networks." *Journal of Scientific Computing*, 92, 88.

> 延伸：將平板幾何改為**球形顆粒**（ $\dfrac{1}{\xi^2}\dfrac{d}{d\xi}(\xi^2\dfrac{dC}{d\xi})=\phi^2 C$ ）、加入**非一級動力學**或**非等溫效應**，都可用相同 PINN 框架延伸。